# Score baseline — GPT-4.1 PF/SC (reuses OmniContext's scorer)

CPU runtime (API calls, no GPU). Runs OmniContext's `OmniContextScore` over the 488 held-out rows — prompt and parsing unchanged; retry loop patched (see 1b). Resumable; scores written to Drive.

## 1. Repo + deps (skip clone if present)

In [1]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","regex"], check=True)   # json_util imports `regex`
sys.path.insert(0, REPO)
os.chdir(REPO)
print("ready")

ready


## 1b. Patch OmniContextScore.evaluate
Upstream retry loop has no `break` → 3 attempts always → 6 API calls/row. Patch cuts it to 2.

In [2]:
p = "/content/OmniGen2/omnicontext/omnicontext_score.py"
lines = open(p).read().split("\n")
if not any("added: stop after successful parse" in l for l in lines):
    out = []
    for ln in lines:
        out.append(ln)
        if 'raise Exception("rate_limit_exceeded")' in ln:
            out.append("                break   # added: stop after successful parse")
    open(p, "w").write("\n".join(out))
    print("patched")
else:
    print("already patched")

patched


In [3]:
import inspect
from omnicontext.omnicontext_score import OmniContextScore
print(inspect.getsource(OmniContextScore.evaluate))

    def evaluate(self, input_image_paths, instruction, with_scene=False):
        results_dict = {}

        max_tries = 3
        PF_scores = None
        SC_scores = None
        for try_idx in range(max_tries):
            try:
                PF_prompt = self.prompt_generator(instruction, task_type="prompt_following")
                SC_prompt = self.prompt_generator(instruction, task_type="subject_consistency", with_scene=with_scene)

                PF_results = ask_gpt4o(input_image_paths, PF_prompt, self.openai_url, self.openai_key)
                SC_results = ask_gpt4o(input_image_paths, SC_prompt, self.openai_url, self.openai_key)

                PF_scores = mllm_output_to_dict(PF_results)
                SC_scores = mllm_output_to_dict(SC_results)

                if PF_scores == "rate_limit_exceeded" or SC_scores == "rate_limit_exceeded":
                    raise Exception("rate_limit_exceeded")
                break   # added: stop after successful parse
            exc

## 2. Mount Drive, key, data

In [4]:
from google.colab import drive, userdata
import os, subprocess
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
GEN   = f"{DRIVE}/gen/baseline"
IMGS, EXPECTED = "/content/imgs", 37063
TAR   = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _,_,f in os.walk(d)) if os.path.isdir(d) else 0
if n_files(IMGS) < EXPECTED:   # fresh runtime -> restore refs; same session -> skip
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
print("reference images:", n_files(IMGS))
if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"])
print("baseline images:", len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else "MISSING")

Mounted at /content/drive
reference images: 37063
baseline images: 488


## 3. Confirm judge model (gpt-4.1)

In [5]:
import inspect
from omnicontext import openai_util
print(inspect.getsource(openai_util.ask_gpt4o))

def ask_gpt4o(image_path, prompt, url, api_key):
    prompt = prepare_prompt(image_path, prompt)
    payload = {
        "model": "gpt-4.1",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1400,
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    try:
        response = requests.post(url, json=payload, headers=headers, timeout=180)  # Set timeout to 5 minutes (300 seconds)
    except Exception as e:
        print(f"Error: {e}")
        return ""
    return extract_response(response)



## Appendix A — locate OmniContext's scoring + aggregation scripts

In [6]:
!ls /content/OmniGen2/omnicontext/
!grep -rln "combined_score\|geometric\|gmean\|overall\|Overall\|\.mean" /content/OmniGen2/omnicontext/*.py

calculate_statistics.py  omnicontext_score.py  __pycache__
inference.py		 openai_util.py        README.md
json_util.py		 prompt_generator.py   test_omnicontext_score.py
/content/OmniGen2/omnicontext/calculate_statistics.py
/content/OmniGen2/omnicontext/prompt_generator.py
/content/OmniGen2/omnicontext/test_omnicontext_score.py


## 3b. Smoke test — 3 rows
Requires generation complete (baseline images: 488).

In [7]:
import os, json, math
from PIL import Image
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]

def resize_ref(im):
    w, h = im.size
    if w*h > 1024*1024:
        r = math.sqrt(1024*1024/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

for i in range(3):
    r = rows[i]
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen = Image.open(f"{GEN}/test_{i:05d}.png").convert("RGB")
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)
    print(f"row {i}: PF={res['PF_scores']['score']}  SC={res['SC_scores']['score']}")
    print(f"   PF: {str(res['PF_scores']['reasoning'])[:140]}")
    print(f"   SC: {str(res['SC_scores']['reasoning'])[:140]}\n")

row 0: PF=6  SC=9
   PF: The image depicts a young man in a sharp suit at a conference table, working intently on a modern laptop. He is dressed professionally, and 
   SC: The man in the final image has highly consistent facial and head features with the individual in the reference image: the same straight, nea

row 1: PF=4  SC=0
   PF: The image depicts a young man in a sharp suit at a conference table, working intently on a laptop, which matches much of the instruction inc
   SC: The editing instruction requires that the laptop in the scene match the one in image 1, specifically featuring its distinctive stickers. How

row 2: PF=9  SC=9
   PF: The final image closely follows the editing instruction: The young man is seated at a sleek conference table, intently working on a modern l
   SC: The man in the final image matches the subject from the original image in nearly all significant facial and head features: hair color and st



## Appendix B — official aggregation formula (Overall = per-item sqrt(PF*SC), then mean)

In [8]:
!cat /content/OmniGen2/omnicontext/calculate_statistics.py

import megfile
import os
import pandas as pd
from collections import defaultdict
import sys
import numpy as np
import math
import json
import glob

def analyze_scores(json_lines, language):    
    group_prompt_following_scores = {}
    group_subject_consistency_scores = {}
    group_overall_scores = {}

    for task_type in json_lines.keys():
        prompt_following_scores = []
        subject_consistency_scores = []
        overall_scores = []

        for json_line in json_lines[task_type]:
            if json_line['instruction_language'] != language:
                continue

            prompt_following_score = json_line['PF_score']
            subject_consistency_score = json_line['SC_score']
            overall_score = math.sqrt(prompt_following_score * subject_consistency_score)
            
            prompt_following_scores.append(prompt_following_score)
            subject_consistency_scores.append(subject_consistency_score)
            overall_scores.append(overall_score)

## 4. Score held-out (resumable)

In [9]:
import os, json, math, time
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from omnicontext.omnicontext_score import OmniContextScore

scorer = OmniContextScore("https://api.openai.com/v1/chat/completions", userdata.get("OPENAI_API_KEY"))
SCORES = f"{DRIVE}/scores/test__baseline.jsonl"
os.makedirs(os.path.dirname(SCORES), exist_ok=True)

def resize_ref(im):   # OmniContext default: cap at 1024^2, round to /16 — references only
    w, h = im.size
    cap = 1024*1024
    if w*h > cap:
        r = math.sqrt(cap/(w*h)); w, h = int(w*r), int(h*r)
    return im.resize((max(16, w//16*16), max(16, h//16*16)))

def num(x):           # fix_json can repair a bare score into [8]
    if isinstance(x, list) and x: x = x[0]
    return x if isinstance(x, (int, float)) else None

rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]

done = set()
if os.path.exists(SCORES):
    for l in open(SCORES, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])   # failures stay retryable
        except: pass

def score_row(i):
    r = rows[i]
    gp = f"{GEN}/test_{i:05d}.png"
    if not os.path.exists(gp): return {"i": i, "error": "missing_generated"}
    refs = [resize_ref(Image.open(os.path.join(IMGS, p)).convert("RGB")) for p in r["input_images"]]
    gen  = Image.open(gp).convert("RGB")            # official: generated image not resized
    time.sleep(0.2)
    res = scorer.evaluate(refs + [gen], r["instruction"], with_scene=False)   # original instruction, never the augmented one
    pf, sc = num(res["PF_scores"]["score"]), num(res["SC_scores"]["score"])
    if pf is None or sc is None: return {"i": i, "error": "unparsable_score"}
    ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
    return {"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])), "n_refs": len(r["input_images"]),
            "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
            "PF": pf, "SC": sc,
            "PF_reason": res["PF_scores"]["reasoning"], "SC_reason": res["SC_scores"]["reasoning"]}

todo = [i for i in range(len(rows)) if i not in done]
print(f"to score: {len(todo)} (done: {len(done)})")
with open(SCORES, "a", encoding="utf-8") as f, ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(score_row, i) for i in todo]
    for n, fut in enumerate(as_completed(futs), 1):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n"); f.flush()
        if n % 25 == 0: print(f"  {n}/{len(todo)}")
print("scoring done ->", SCORES)

to score: 0 (done: 488)
scoring done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/test__baseline.jsonl


## 5. Aggregate — PF / SC / Overall, by reference count and provenance

In [10]:
import json, math
from collections import defaultdict
SCORES = f"{DRIVE}/scores/test__baseline.jsonl"

recs, errs = [], defaultdict(int)
for l in open(SCORES, encoding="utf-8"):
    d = json.loads(l)
    if "error" in d: errs[d["error"]] += 1; continue
    recs.append(d)
guessed = sum(1 for d in recs if "guess_if_cannot_parse" in str(d["PF_reason"]) + str(d["SC_reason"]))

def agg(rs):   # Overall = per-item sqrt(PF*SC), then mean  (calculate_statistics.py)
    pf = [r["PF"] for r in rs]; sc = [r["SC"] for r in rs]
    ov = [math.sqrt(p*s) for p, s in zip(pf, sc)]
    return sum(pf)/len(pf), sum(sc)/len(sc), sum(ov)/len(ov)

def table(key):
    g = defaultdict(list)
    for r in recs: g[r[key]].append(r)
    for k in sorted(g, key=str):
        p, s, o = agg(g[k]); print(f"  {str(k):8s} n={len(g[k]):3d}  PF={p:.3f}  SC={s:.3f}  Overall={o:.3f}")

p, s, o = agg(recs)
print(f"n={len(recs)}  errors={dict(errs)}  guessed={guessed}")
print(f"POOLED   PF={p:.3f}  SC={s:.3f}  Overall={o:.3f}\n")
print("by reference-subject count:"); table("n_refs")
print("\nby reference provenance:");   table("ref_kind")

n=488  errors={}  guessed=0
POOLED   PF=8.004  SC=8.088  Overall=7.963

by reference-subject count:
  1        n=243  PF=8.181  SC=8.058  Overall=7.994
  2        n=194  PF=7.851  SC=8.155  Overall=7.958
  3        n= 51  PF=7.745  SC=7.980  Overall=7.832

by reference provenance:
  all_up   n=198  PF=7.919  SC=7.424  Overall=7.520
  mixed    n=133  PF=8.038  SC=8.436  Overall=8.205
  no_up    n=157  PF=8.083  SC=8.631  Overall=8.317
